<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/19_transformer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Transformer from Scratch in PyTorch

> **Track:** ML Engineer | **Level:** Advanced | **Time:** 4-6 hours

## Overview
The Transformer ("Attention Is All You Need", Vaswani et al. 2017) is the architecture behind GPT, BERT, and every modern LLM. Building it from scratch gives deep intuition for what frameworks do under the hood.

### What You'll Learn
- Scaled dot-product attention
- Multi-head attention mechanism
- Positional encoding
- Feed-forward sublayer with LayerNorm
- Encoder and decoder blocks
- Full seq2seq Transformer
- Training on a toy translation task

```bash
pip install torch numpy matplotlib
```

In [ ]:
# Setup: imports and scaled dot-product attention
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

def scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    mask: torch.Tensor | None = None
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute scaled dot-product attention.
    Args: Q, K, V each of shape (batch, heads, seq_len, d_k)
    Returns: output (batch, heads, seq_len, d_v), attention_weights
    """
    d_k = query.size(-1)
    scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)  # (batch, heads, seq, seq)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = attn_weights @ value
    return output, attn_weights

# Test attention
B, H, S, D = 2, 4, 8, 16  # batch, heads, seq_len, d_k
Q = torch.randn(B, H, S, D)
K = torch.randn(B, H, S, D)
V = torch.randn(B, H, S, D)
out, attn = scaled_dot_product_attention(Q, K, V)
print(f'Attention output shape: {out.shape}')
print(f'Attention weights shape: {attn.shape} (sums to 1 per row: {attn.sum(-1).mean():.4f})')

## 1. Multi-Head Attention and Positional Encoding

In [ ]:
# Multi-Head Attention module
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, S, D = x.shape
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask=None):
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask)
        B, H, S, D = attn_out.shape
        concat = attn_out.transpose(1, 2).contiguous().view(B, S, H * D)
        return self.W_o(concat), weights

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(x + self.pe[:, :x.size(1)])

print('MultiHeadAttention and PositionalEncoding defined.')

## 2. Encoder and Decoder Blocks

In [ ]:
# Encoder block: self-attention + feed-forward
class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask=None):
        attn_out, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        return self.norm2(x + self.dropout(ff_out))

# Decoder block: self-attention + cross-attention + feed-forward
class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor, src_mask=None, tgt_mask=None):
        sa_out, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(sa_out))
        ca_out, _ = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + self.dropout(ca_out))
        ff_out = self.ff(x)
        return self.norm3(x + self.dropout(ff_out))

print('Encoder and Decoder blocks defined.')

## 3. Full Transformer Model

In [ ]:
# Complete Transformer (seq2seq)
class Transformer(nn.Module):
    def __init__(self, src_vocab: int, tgt_vocab: int, d_model: int = 128, num_heads: int = 4,
                 num_layers: int = 2, d_ff: int = 256, max_len: int = 100, dropout: float = 0.1):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.encoder = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, tgt_vocab)
        self.scale = math.sqrt(d_model)

    def encode(self, src: torch.Tensor, src_mask=None) -> torch.Tensor:
        x = self.pos_enc(self.src_embed(src) * self.scale)
        for layer in self.encoder:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt: torch.Tensor, enc_out: torch.Tensor, src_mask=None, tgt_mask=None) -> torch.Tensor:
        x = self.pos_enc(self.tgt_embed(tgt) * self.scale)
        for layer in self.decoder:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.fc_out(x)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor, src_mask=None, tgt_mask=None) -> torch.Tensor:
        enc_out = self.encode(src, src_mask)
        return self.decode(tgt, enc_out, src_mask, tgt_mask)

SRC_VOCAB, TGT_VOCAB = 1000, 1000
model = Transformer(SRC_VOCAB, TGT_VOCAB)
total_params = sum(p.numel() for p in model.parameters())
print(f'Transformer model: {total_params:,} parameters')

# Test forward pass
src = torch.randint(0, SRC_VOCAB, (2, 10))  # batch=2, seq=10
tgt = torch.randint(0, TGT_VOCAB, (2, 8))
out = model(src, tgt)
print(f'Output shape: {out.shape} (batch, tgt_seq, vocab_size)')

## 4. Training on a Toy Task

In [ ]:
# Train on a simple copy task: output should equal input
from torch.optim import Adam

VOCAB = 20
SEQ_LEN = 10
BATCH_SIZE = 64

toy_model = Transformer(VOCAB, VOCAB, d_model=64, num_heads=4, num_layers=2, d_ff=128)
optimizer = Adam(toy_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

def generate_copy_batch(batch_size: int, seq_len: int, vocab: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate (src, tgt) pairs where tgt = src (copy task)."""
    src = torch.randint(1, vocab, (batch_size, seq_len))
    return src, src

train_losses = []
for step in range(200):
    toy_model.train()
    src, tgt = generate_copy_batch(BATCH_SIZE, SEQ_LEN, VOCAB)
    tgt_in = tgt[:, :-1]   # decoder input (shifted right)
    tgt_out = tgt[:, 1:]   # expected output
    logits = toy_model(src, tgt_in)
    loss = criterion(logits.reshape(-1, VOCAB), tgt_out.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(toy_model.parameters(), 1.0)
    optimizer.step()
    if step % 50 == 0:
        train_losses.append(loss.item())
        print(f'Step {step:4d} | Loss: {loss.item():.4f}')

# Check if it learned the copy task
toy_model.eval()
with torch.no_grad():
    test_src, _ = generate_copy_batch(4, SEQ_LEN, VOCAB)
    logits = toy_model(test_src, test_src[:, :-1])
    preds = logits.argmax(-1)
    exact = (preds == test_src[:, 1:]).float().mean().item()
    print(f'\nCopy task token accuracy: {exact:.1%}')

## Exercises

1. **Attention visualization**: Extract the attention weights from the last encoder block during inference. Use matplotlib to plot a heatmap of the attention pattern (x-axis: key positions, y-axis: query positions). What patterns emerge for the copy task?

2. **Causal mask**: Implement `make_causal_mask(seq_len)` that creates a lower-triangular boolean mask (True = can attend, False = masked). Apply it to the decoder self-attention and verify that the model can't attend to future tokens.

3. **Beam search decoding**: Implement beam search (beam width k=3) for the toy Transformer. At each step, keep the top-k partial sequences by log-probability. Compare the quality of beam search vs greedy decoding on the copy task.